# 00 · Primeiro contato: previsão de doença cardíaca

**Entrega 1** · Machine Learning, Projeto Final

**Grupo:** _preencher nomes dos integrantes_

Este notebook cobre o que a Entrega 1 pede (itens 2 a 6) e já deixa a base pronta para as próximas entregas:

| Item da Entrega 1 | Onde está |
| --- | --- |
| Problema em uma frase | Seção 1 |
| Tipo e alvo | Seção 1 |
| Dataset: link, licença, linhas × colunas, dicionário | Seção 2 |
| Métrica principal justificada e resultado do baseline | Seções 5 e 6 |
| Esboço do fluxo formulário → API → resposta | Seção 7 |

**Regra do teste trancado:** a divisão treino/teste é feita logo depois de carregar os dados. O teste (20%) **não é usado em nenhum cálculo deste notebook**; ele só será aberto na Entrega 3. Toda exploração, validação e comparação de baselines usa apenas o treino.

## 1. Problema, tipo e alvo

> **Dado** o perfil clínico, a idade, o colesterol e a pressão arterial de um paciente, **prever** a ocorrência de doença cardíaca **para que** a equipe médica possa priorizar exames preventivos.

- **Tipo:** classificação binária.
- **Alvo:** `HeartDisease` (1 = tem doença cardíaca, 0 = não tem).
- **Classe positiva:** `1`, a doença. É nela que a métrica principal é medida.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, make_scorer, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# ---- Parâmetros do projeto ----
RANDOM_STATE = 42          # reprodutibilidade: mesmo resultado em qualquer computador
ALVO = "HeartDisease"
ARQUIVO = None             # None = procura heart*.csv; ou fixe, ex.: Path("../data/heart_1.csv")

# Piso de especificidade (decisão do grupo, ajustável): sem ele, "marcar todo mundo como doente"
# teria recall perfeito. Especificidade = fração dos pacientes SEM doença que o modelo libera.
ESPECIFICIDADE_MIN = 0.70

# ---- Colunas ----
# Entram só variáveis que o usuário consegue informar no formulário.
NUMERICAS = ["Age", "RestingBP", "Cholesterol", "MaxHR"]
BINARIAS = ["FastingBS"]
CATEGORICAS = ["Sex", "ChestPainType", "ExerciseAngina"]
FEATURES = NUMERICAS + BINARIAS + CATEGORICAS
# Decisão do grupo: ficam de fora por dependerem de exame clínico especializado.
EXCLUIDAS = ["RestingECG", "Oldpeak", "ST_Slope"]

# Cores consistentes nos gráficos (par validado para daltonismo)
COR_SEM, COR_COM = "#1f6fb4", "#d9730d"

## 2. Dataset

- **Nome:** Heart Failure Prediction Dataset (Kaggle). _Confirmar o autor e colar aqui o link exato da página._
- **Link da fonte:** _preencher_
- **Licença:** _preencher com a licença que aparece na página do Kaggle e confirmar que permite o uso._
- **Linhas × colunas:** conferido na célula de carga abaixo.

**Dicionário das colunas de entrada** (conferir com a descrição da página do Kaggle):

| Coluna | Significado | Valores |
| --- | --- | --- |
| `Age` | Idade | anos |
| `Sex` | Sexo | `M` ou `F` |
| `ChestPainType` | Tipo de dor no peito | `TA` angina típica, `ATA` angina atípica, `NAP` dor não anginosa, `ASY` assintomático |
| `RestingBP` | Pressão arterial em repouso | mm Hg |
| `Cholesterol` | Colesterol sérico | mg/dl |
| `FastingBS` | Glicemia em jejum acima de 120 mg/dl | `1` sim, `0` não |
| `MaxHR` | Frequência cardíaca máxima atingida | bpm |
| `ExerciseAngina` | Dor no peito induzida por exercício | `Y` ou `N` |
| `HeartDisease` | **Alvo**: doença cardíaca | `1` ou `0` |

**Colunas deixadas de fora** (`RestingECG`, `Oldpeak`, `ST_Slope`): dependem de exame especializado, e o usuário não consegue informá-las num formulário.

In [ ]:
def localizar_csv():
    if ARQUIVO is not None:
        return Path(ARQUIVO)
    for pasta in [Path("../data"), Path("data"), Path("dados"), Path(".")]:
        achados = sorted(pasta.glob("heart*.csv")) if pasta.exists() else []
        if achados:
            return achados[0]
    return None


caminho = localizar_csv()
if caminho is None:
    try:
        # Colab: abre o seletor de arquivos para enviar o CSV
        from google.colab import files
        print("CSV não encontrado. Selecione o arquivo do dataset:")
        enviado = files.upload()
        nome = next(iter(enviado))
        Path("data").mkdir(exist_ok=True)
        caminho = Path("data") / nome
        caminho.write_bytes(enviado[nome])
    except ImportError:
        raise FileNotFoundError(
            "Não achei o CSV. Coloque o arquivo (nome começando com 'heart') em data/ "
            "ou ajuste ARQUIVO na célula de configuração."
        )

df = pd.read_csv(caminho)
print(f"Arquivo: {caminho}")
print(f"Linhas × colunas: {df.shape[0]:,} × {df.shape[1]}")

faltando = [c for c in FEATURES + [ALVO] if c not in df.columns]
if faltando:
    raise KeyError(f"Colunas esperadas que não existem no arquivo: {faltando}\nColunas do arquivo: {list(df.columns)}")

df.head()

## 3. Limpeza mínima e divisão (teste trancado)

**Zeros impossíveis.** Colesterol e pressão em repouso iguais a `0` não existem em paciente vivo; são dado ausente codificado como zero. Eles viram `NaN` e serão preenchidos **dentro do pipeline**, com a mediana aprendida só no treino. Esta troca de `0` por `NaN` é uma regra fixa (não aprende nada dos dados), então pode ser feita antes da divisão.

**Divisão 80/20 estratificada** (`stratify=y`), com semente fixa, para treino e teste terem a mesma proporção de cada classe.

In [ ]:
zeros = {c: int((df[c] == 0).sum()) for c in ["Cholesterol", "RestingBP"]}
print("Valores iguais a 0 (tratados como ausentes):", zeros)

df = df.copy()
for c in ["Cholesterol", "RestingBP"]:
    df.loc[df[c] == 0, c] = np.nan

X = df[FEATURES]
y = df[ALVO]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"\nTreino: {len(X_treino):,} linhas | Teste (trancado): {len(X_teste):,} linhas")
print("Proporção de doença  ->  treino: {:.1%} | teste: {:.1%}".format(y_treino.mean(), y_teste.mean()))

# A partir daqui, X_teste e y_teste NÃO são usados até a Entrega 3.
treino = X_treino.assign(**{ALVO: y_treino})

## 4. Exploração (somente treino)

In [ ]:
print("Tipos e ausentes (treino):")
resumo = pd.DataFrame({
    "tipo": treino.dtypes.astype(str),
    "ausentes": treino.isna().sum(),
    "% ausente": treino.isna().mean() * 100,
    "valores distintos": treino.nunique(),
})
display(resumo)

print("\nBalanceamento do alvo (treino):")
display(
    treino[ALVO].map({0: "sem doença", 1: "com doença"}).value_counts().to_frame("pacientes")
    .assign(percentual=lambda t: t["pacientes"] / t["pacientes"].sum() * 100)
)

print("\nResumo das variáveis numéricas:")
display(treino[NUMERICAS].describe().T)

In [ ]:
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
fig, eixos = plt.subplots(1, 3, figsize=(16, 4.2))

# (a) Taxa de doença por tipo de dor no peito
taxa = treino.groupby("ChestPainType")[ALVO].agg(["mean", "size"]).sort_values("mean")
barras = eixos[0].barh(taxa.index, taxa["mean"] * 100, color=COR_COM)
for barra, (_, linha) in zip(barras, taxa.iterrows()):
    eixos[0].text(barra.get_width() + 1, barra.get_y() + barra.get_height() / 2,
                  f"{linha['mean']:.0%}  (n={int(linha['size'])})", va="center")
eixos[0].set_xlim(0, 115)
eixos[0].set_xlabel("% de pacientes com doença")
eixos[0].set_title("Doença por tipo de dor no peito")

# (b) Distribuição da idade por classe
bins = np.arange(25, 85, 5)
eixos[1].hist(treino.loc[treino[ALVO] == 0, "Age"], bins=bins, alpha=0.65, color=COR_SEM, label="Sem doença")
eixos[1].hist(treino.loc[treino[ALVO] == 1, "Age"], bins=bins, alpha=0.65, color=COR_COM, label="Com doença")
eixos[1].set_xlabel("Idade (anos)")
eixos[1].set_ylabel("Pacientes")
eixos[1].set_title("Idade por classe")
eixos[1].legend(frameon=False)

# (c) Colesterol por classe (somente valores válidos)
valido = treino.dropna(subset=["Cholesterol"])
bins_c = np.arange(80, 620, 40)
eixos[2].hist(valido.loc[valido[ALVO] == 0, "Cholesterol"], bins=bins_c, alpha=0.65, color=COR_SEM, label="Sem doença")
eixos[2].hist(valido.loc[valido[ALVO] == 1, "Cholesterol"], bins=bins_c, alpha=0.65, color=COR_COM, label="Com doença")
eixos[2].set_xlabel("Colesterol (mg/dl)")
eixos[2].set_title(f"Colesterol por classe ({treino['Cholesterol'].isna().sum()} ausentes fora do gráfico)")
eixos[2].legend(frameon=False)

plt.tight_layout()
plt.show()

## 5. Métrica principal

**Escolha: recall da classe "doença"** (sensibilidade), com um piso de especificidade.

**Por quê.** O modelo existe para priorizar exames preventivos. Deixar passar um paciente doente (falso negativo) é mais grave do que chamar um paciente saudável para um exame a mais (falso positivo). O grupo concorda em aceitar mais alarmes falsos para perder menos doentes.

**Por que só o recall não basta.** Como 55% dos pacientes têm doença, um "modelo" que responde *sempre doente* tem recall 1,00 sem aprender nada. Por isso a métrica vem com um **piso de especificidade** (`ESPECIFICIDADE_MIN`): só vale um modelo com recall alto *e* especificidade acima do piso.

| Métrica | Papel |
| --- | --- |
| **Recall da doença** | Principal: de cada 100 doentes, quantos o modelo encontra |
| **Especificidade** | Restrição: de cada 100 saudáveis, quantos o modelo libera (≥ piso) |
| Precisão | De quem foi marcado como doente, quantos realmente são |
| F1 | Equilíbrio entre precisão e recall |
| ROC AUC | Qualidade do ranqueamento, independe do limiar de decisão |

A acurácia não é usada como critério porque trata os dois tipos de erro como se custassem o mesmo.

In [ ]:
pontuacao = {
    "recall": "recall",
    "especificidade": make_scorer(recall_score, pos_label=0),
    "precisao": "precision",
    "f1": "f1",
    "auc": "roc_auc",
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 6. Baselines

Três referências, do mais simples ao mais forte. Todas são avaliadas por **validação cruzada estratificada de 5 partições, só no treino**. O pré-processamento fica dentro do `Pipeline`, então a mediana, a escala e o encoding são aprendidos em cada partição apenas com os dados de treino dela, sem vazamento.

1. **Dummy:** responde sempre a classe mais frequente. Mostra o que se consegue sem aprender nada.
2. **Regra de uma variável:** árvore de decisão com uma única divisão. O modelo escolhe sozinho a melhor pergunta.
3. **Regressão logística:** baseline forte e interpretável; é a referência que um modelo mais complexo precisa superar.

In [ ]:
pre = ColumnTransformer([
    ("numericas", Pipeline([
        ("imputar", SimpleImputer(strategy="median", add_indicator=True)),
        ("padronizar", StandardScaler()),
    ]), NUMERICAS),
    ("binarias", "passthrough", BINARIAS),
    ("categoricas", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), CATEGORICAS),
])

classe_maioria = y_treino.mode()[0]
baselines = {
    f"1. Dummy (sempre {'doença' if classe_maioria == 1 else 'sem doença'})":
        DummyClassifier(strategy="most_frequent"),
    "2. Regra de uma variável (árvore de 1 divisão)":
        Pipeline([("pre", pre), ("modelo", DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE))]),
    "3. Regressão logística":
        Pipeline([("pre", pre), ("modelo", LogisticRegression(max_iter=1000))]),
}

medias, desvios = {}, {}
for nome, modelo in baselines.items():
    r = cross_validate(modelo, X_treino, y_treino, cv=cv, scoring=pontuacao)
    medias[nome] = {m: r[f"test_{m}"].mean() for m in pontuacao}
    desvios[nome] = {m: r[f"test_{m}"].std() for m in pontuacao}

medias = pd.DataFrame(medias).T
desvios = pd.DataFrame(desvios).T

tabela = medias.copy().astype(object)
for m in pontuacao:
    tabela[m] = [f"{medias.loc[n, m]:.2f} ± {desvios.loc[n, m]:.2f}" for n in medias.index]
tabela["atende o piso?"] = np.where(medias["especificidade"] >= ESPECIFICIDADE_MIN, "sim", "NÃO")
tabela.columns = ["recall (principal)", "especificidade", "precisão", "F1", "ROC AUC", "atende o piso?"]

print(f"Validação cruzada (5 partições) no treino · piso de especificidade = {ESPECIFICIDADE_MIN:.2f}")
tabela

In [ ]:
# Qual é a "regra de uma variável" que a árvore aprendeu no treino inteiro?
arvore = baselines["2. Regra de uma variável (árvore de 1 divisão)"].fit(X_treino, y_treino)
nomes = arvore.named_steps["pre"].get_feature_names_out()
print(export_text(arvore.named_steps["modelo"], feature_names=list(nomes)))

In [ ]:
# Matriz de confusão da regressão logística (previsões de validação cruzada, só no treino)
logistica = baselines["3. Regressão logística"]
previsto = cross_val_predict(logistica, X_treino, y_treino, cv=cv)

fig, ax = plt.subplots(figsize=(4.6, 4))
ConfusionMatrixDisplay.from_predictions(
    y_treino, previsto, display_labels=["sem doença", "com doença"],
    cmap="Blues", colorbar=False, ax=ax,
)
ax.set_title("Regressão logística (validação cruzada no treino)")
ax.set_xlabel("Previsto")
ax.set_ylabel("Real")
plt.tight_layout()
plt.show()

fn = int(((y_treino == 1) & (previsto == 0)).sum())
fp = int(((y_treino == 0) & (previsto == 1)).sum())
print(f"Doentes que passaram sem alerta (falsos negativos): {fn}")
print(f"Saudáveis com alarme falso (falsos positivos):      {fp}")

**Leitura do resultado.** O Dummy "acerta" todos os doentes, mas libera zero saudáveis, então falha no piso de especificidade e não serve. O modelo que o grupo precisa superar na Entrega 3 é a **regressão logística**: o que importa é aumentar o recall sem deixar a especificidade cair abaixo do piso.

**Próximo passo de modelagem (Entrega 3):** com a regressão logística (e modelos mais fortes), escolher o limiar de decisão para favorecer o recall, já que aceitamos mais alarmes falsos. O limiar deve ser escolhido com validação cruzada no treino; o teste só entra no final.

## 7. Esboço do fluxo: formulário → API → resposta

```
┌────────────┐   JSON    ┌───────────────────┐   probabilidade   ┌─────────────┐
│ Formulário │ ────────► │ API (FastAPI)     │ ────────────────► │  Resposta   │
│ (interface)│           │ valida os campos  │                   │ ao usuário  │
└────────────┘           │ e chama o modelo  │                   └─────────────┘
                         └───────────────────┘
                           pipeline.joblib
```

**O que entra** (o que o usuário consegue informar):

| Campo | Tipo | Faixa / valores |
| --- | --- | --- |
| `age` | inteiro | 18 a 100 |
| `sex` | texto | `M` ou `F` |
| `chest_pain_type` | texto | `TA`, `ATA`, `NAP`, `ASY` |
| `resting_bp` | inteiro | pressão em repouso, mm Hg |
| `cholesterol` | inteiro | mg/dl |
| `fasting_bs` | 0 ou 1 | glicemia em jejum acima de 120 mg/dl |
| `max_hr` | inteiro | bpm |
| `exercise_angina` | texto | `Y` ou `N` |

**O que sai** (o que o usuário vê):

```json
{
  "modelo": "regressao_logistica",
  "versao": "0.1.0",
  "probabilidade_doenca": 0.82,
  "classificacao": "risco elevado",
  "mensagem": "Recomenda-se priorizar exames preventivos. Esta ferramenta não substitui avaliação médica."
}
```

A faixa exata de cada campo e o limiar que define "risco elevado" serão fechados com a modelagem.

## Limites e uso responsável

- O dataset tem poucas centenas de pacientes e não é brasileiro; os resultados **não** devem ser tratados como válidos para a população do Brasil.
- Há mais homens que mulheres na base, então o desempenho pode ser pior para mulheres. Reportar o erro por sexo na Entrega 3.
- A ferramenta **apoia a priorização** de exames; ela não diagnostica e não substitui avaliação médica.
- Os valores `0` em colesterol e pressão foram tratados como dado ausente; essa decisão deve constar no README.

In [ ]:
import json
import joblib
import sklearn
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

# Modelo provisório: preparação + regressão logística, treinado só no treino
modelo_final = make_pipeline(pre, LogisticRegression(max_iter=1000))
modelo_final.fit(X_treino, y_treino)

Path("../models").mkdir(exist_ok=True)
joblib.dump(modelo_final, "../models/pipeline.joblib")

metadados = {
    "modelo": "regressao_logistica",
    "versao": "0.1.0-provisorio",
    "limiar": 0.5,                  # provisório: trocar pelo limiar escolhido na modelagem
    "colunas": FEATURES,
    "sklearn": sklearn.__version__,
}
with open("../models/metadados.json", "w") as f:
    json.dump(metadados, f, indent=2)

# Teste: carregar e prever um paciente de exemplo
pipeline = joblib.load("../models/pipeline.joblib")
paciente = pd.DataFrame([{
    "Age": 54, "Sex": "M", "ChestPainType": "ASY", "RestingBP": 140,
    "Cholesterol": 239, "FastingBS": 0, "MaxHR": 120, "ExerciseAngina": "Y",
}])
print("Probabilidade de doença:", round(pipeline.predict_proba(paciente)[0, 1], 2))